In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().resolve().parent))
from src.split import split_data

### Loading dataset

In [ ]:
raw_train_df = pd.read_csv('../data/raw/train.csv')
raw_train_df.head()

In [ ]:
X_train_df, X_val_df, y_train_df, y_val_df = split_data(raw_train_df)
X_train_df.head()

#### Need to understand datatypes

In [ ]:
print(X_train_df.info())
# print(X_train_df.describe())
# print(X_train_df.shape)


### Interpreting missing value. the percentage of the missing value

In [ ]:
isnull =pd.DataFrame((X_train_df.isnull().mean()*100).sort_values(ascending=False))
isnull_true = isnull[isnull[0] > 0]
isnull_true

### Now you have to understand what those missing value means. missing can be meant no info or no access to the feature

In [ ]:
feature_absent= ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual','GarageYrBlt', 'GarageCond', 'BsmtExposure', 'BsmtFinType2', 'BsmtFinType1', 'BsmtCond', 'BsmtQual']
info_with_diff_value = ['MasVnrType', 'LotFrontage', 'MasVnrArea']
missing_means_no_data = ['Electrical'] # it is different bc every hose has electricity
len(feature_absent) + len(info_with_diff_value) + len(missing_means_no_data)


### Intepreting masvanarea and masvantype

In [ ]:
X_train_df[(X_train_df['MasVnrArea'] == 0) | (X_train_df['MasVnrArea'].isnull())].shape[0]


In [ ]:
X_train_df['MasVnrType'].isnull().sum()

### Apparently 3 cases type has more null than area.

In [ ]:
X_train_df.loc[(X_train_df['MasVnrType'].isnull()) & ((X_train_df['MasVnrArea'] == 0) | X_train_df['MasVnrArea'].isnull()),['MasVnrType', 'MasVnrArea']]

#### I need to find out if masvantype not equal to masvanarea which is 0. 

In [ ]:
masvntype = X_train_df.loc[(X_train_df['MasVnrType'].isnull()) & (X_train_df['MasVnrArea'] != 0) & X_train_df['MasVnrArea'].notnull(),['MasVnrType', 'MasVnrArea']]
masvanarea = X_train_df.loc[X_train_df['MasVnrArea'].isnull() ,['MasVnrType', 'MasVnrArea']]
masvntype

In [ ]:
X_train_df.loc[
    (X_train_df["MasVnrArea"] < 20 )& (X_train_df['MasVnrArea'] != 0),
    ["MasVnrArea", "MasVnrType"]
]

In [ ]:
X_train_df.groupby('MasVnrType')['MasVnrArea'].median()

### It means in 5 occassions masvnrtype is null but masvnrarea was there

In [ ]:
masvanarea2 = X_train_df.loc[(X_train_df['MasVnrArea'].isnull() | X_train_df['MasVnrArea'] == 0) & (X_train_df['MasVnrType'].notnull()),['MasVnrType', 'MasVnrArea']]
masvanarea2

### in two occasions masvnrtype was not null but masvnrarea was 0

### My initial decision was that i will find the average area of each type and replace the inconsistency. Now if those values overlap very often i can't replace them with average. Now i have to test my hypothesis.

In [ ]:
sns.boxplot(x='MasVnrType', y='MasVnrArea', data=X_train_df)

## INTERPRETING THE BOXPLOT : 
#### Boxplot shows that distribution is quite similar. it means the median, mean mode are probably around at the same value. it also shows outliers exist in brkface and stone where brkcmn has less outlier.

In [ ]:
X_train_df['MasVnrType'].value_counts()

In [ ]:
X_train_df['MasVnrArea'].value_counts()

In [ ]:
sns.histplot(data=X_train_df, x='MasVnrArea', bins=30, kde=True)

### 

#### Now diving into 'lotfrontage'

In [ ]:
X_train_df['LotFrontage'].isnull().sum()

In [ ]:
sns.boxplot(data=X_train_df, x='LotFrontage')

### 217 missing values . think which features can help us to get lotfrontage.

In [ ]:
sns.scatterplot(data=X_train_df, x='LotFrontage', y='LotArea')

In [ ]:
sns.boxplot(x='Neighborhood', y='LotFrontage', data=X_train_df)
plt.xticks(rotation = 90)
plt.tight_layout()
plt.show()

### to find the missing value of the lotfrontage i will use neighbourhoods leftfrontage individually as in the same neighborhoods houses tends to have similar lotfrontage. but what should i use mean or medium.

In [ ]:
X_train_df['Neighborhood'].value_counts(normalize=True)

In [ ]:
X_train_df['Neighborhood'].isnull().sum()

## No null values for 'Neighborhood'

In [ ]:
lotfront = X_train_df.loc[X_train_df['LotFrontage'].isnull(), ['LotFrontage','Neighborhood']]
lotfront['Neighborhood'].value_counts(normalize=True)

In [ ]:
X_train_df['Neighborhood'].value_counts()

In [ ]:
sns.displot(data=X_train_df, x='LotFrontage', col='Neighborhood', col_wrap= 3,col_order=(X_train_df['Neighborhood'].value_counts().index),kde = True)
plt.xticks(rotation = 90)
plt.tight_layout()
plt.show()

#### apparently a lot of the neighborhoods has outlier that influence the shape. hence using median instead of mean is more robust and some of the neighborhood's dont contain a lot of houses in that case i will use overall median. now i set the threshold at 20 for now. later i will evaluate if that makes any changes to the model.

### Electrical

In [ ]:
X_train_df['Electrical'].value_counts(normalize=True)

In [ ]:
X_train_df['Electrical'].isnull().sum()

### apparently electrical has only one missing value. and 91 percent of the values are piled in the same value. Using mode is more sensible as it is also categorical

In [ ]:
X_train_df.isnull().sum()

#### which feature needs to be normalize or see the distribution find the skewness 

In [ ]:
# numerical features
numerical_features = X_train_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train_df.select_dtypes(include=[str]).columns.tolist()
len(numerical_features), len(categorical_features)

In [ ]:
for i in numerical_features:
    sns.histplot(data=X_train_df, x=i, bins=100, kde=True)
    plt.xticks(rotation = 90)
    plt.tight_layout()
    plt.show()

In [ ]:
for i in numerical_features:
    sns.displot(data=X_train_df, x=i, bins=100, kde=True)
    plt.xticks(rotation = 90)
    plt.tight_layout()
    plt.show()

In [ ]:
for i in numerical_features:
    sns.boxplot(data=X_train_df, x=i)
    plt.xticks(rotation = 90)
    plt.tight_layout()
    plt.show()

In [ ]:
X_train_df.shape[0]

In [ ]:
X_train_df[numerical_features].nunique().sort_values(ascending=False)

In [ ]:
X_train_df[numerical_features].skew().sort_values(ascending=False)


In [ ]:
mild_skew = [i for i in numerical_features if X_train_df[i].skew() <= 1 and X_train_df[i].skew() >= -1]
mild_right_skew = [i for i in numerical_features if X_train_df[i].skew() > 1 and X_train_df[i].skew() <= 3 ]
extreme_right_skew = [i for i in numerical_features if X_train_df[i].skew() > 3]
mild_left_skew = [i for i in numerical_features if X_train_df[i].skew() < -1 and X_train_df[i].skew() >= -3 ]
extreme_left_skew = [i for i in numerical_features if X_train_df[i].skew() < -3]
len(mild_skew), len(mild_right_skew), len(extreme_right_skew), len(mild_left_skew), len(extreme_left_skew)

#### Now finding the outliers

In [ ]:
q1 = X_train_df[numerical_features].quantile(0.25)
q3 = X_train_df[numerical_features].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = (X_train_df[numerical_features] < lower_bound) | (X_train_df[numerical_features] > upper_bound)
outliers_count = outliers.mean()*100
outliers_count = outliers_count.sort_values(ascending=False)
outliers_count

In [ ]:
unusual_outliers = [i if outliers_count[i] > 5 else None for i in outliers_count.index]
unusual_outliers = [i for i in unusual_outliers if i is not None]
unusual_outliers



In [ ]:
for i in unusual_outliers:
    sns.boxplot(data=X_train_df, x=i)
    plt.show()

### apparently some of the feature doesn't have more unique values. we gotta differentiate different numerical values

In [ ]:
len(numerical_features)

### removing id 

In [ ]:
numerical_features.remove('Id')
len(numerical_features)

In [ ]:
unique_values = X_train_df[numerical_features].nunique().sort_values(ascending=False)
unique_values

In [ ]:
numeric_int = X_train_df[numerical_features].select_dtypes(include=[np.int64, np.int32]).columns.tolist()
numeric_float = X_train_df[numerical_features].select_dtypes(include=[np.float64, np.float32]).columns.tolist()
len(numeric_int), len(numeric_float)

In [ ]:
numeric_float

#### All the features that shows only year , i will put them in the temporal category

In [ ]:
temporal = ['GarageYrBlt', 'YearBuilt', 'YearRemodAdd', 'MoSold', 'YrSold']
insufficient_unique_values = [i for i in numerical_features if X_train_df[i].nunique() < 30 and i not in temporal]
insufficient_unique_values, len(insufficient_unique_values)

## continous 

In [ ]:
continuous_features = [i for i in numerical_features if i not in insufficient_unique_values and i not in temporal]
continuous_features, len(continuous_features)

### ordinal

In [ ]:
ordinal_features = ['OverallQual', 'OverallCond', ]
other_continuous_features = ['LowQualFinSF','3SsnPorch','PoolArea', 'MiscVal']
count_features = ['BsmtHalfBath', 'BsmtFullBath', 'HalfBath', 'FullBath', 'TotRmsAbvGrd', 'KitchenAbvGr', 'Fireplaces', 'GarageCars', 'BedroomAbvGr']
extra_categorical_features = ['MSSubClass']
len(ordinal_features) + len(other_continuous_features) + len(count_features) + len(extra_categorical_features)

### The block on top saying other continous feature is those feature are continous but has very less nunique. bc i separated continous values based on nunique. those other_continous_feature has lesss nunique so they are in insufficient_numeric_values. now i have to merge them with continous feature.

### categorical 

In [ ]:
categorical_features, len(categorical_features)

### Below cell contains the untimate features that expresses different value. 

In [ ]:
total_continous_features = continuous_features + other_continuous_features
print('Continous features: ', len(total_continous_features))
print('Temporal features: ', len(temporal))
print('Ordinal features: ', len(ordinal_features))
print('Count features: ', len(count_features))
total_categorical_features = categorical_features + extra_categorical_features
print('Categorical features: ', len(total_categorical_features))
total_features = total_continous_features + temporal + ordinal_features + count_features + total_categorical_features
print('Total features: ', len(total_features))

In [ ]:
X_train_df.shape[1]

### Summation of all the feature equal to the total feature

### Now i will find the outlier

In [ ]:
q1 = X_train_df[total_continous_features].quantile(0.25)
q3 = X_train_df[total_continous_features].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

cont_outliers = (X_train_df[total_continous_features] < lower_bound) | (X_train_df[total_continous_features] > upper_bound)

outliers_count = (cont_outliers.mean()*100).sort_values(ascending=False)
outliers_count


In [ ]:
for i in total_continous_features:
    sns.histplot(data=X_train_df, x=i, kde=True)
    plt.show()

#### A new problem appears XD. dataset contains a lot of zero. that's why out outlier is affected by the zeros . we need to get rid of them first

In [ ]:
non_zero_df = X_train_df[total_continous_features].replace(0, np.nan)
q1 = non_zero_df[total_continous_features].quantile(0.25)
q3 = non_zero_df[total_continous_features].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

final_outliers = (non_zero_df[total_continous_features] < lower_bound) | (non_zero_df[total_continous_features] > upper_bound)
final_outliers_count = (final_outliers.mean()*100).sort_values(ascending=False)
final_outliers_count

In [ ]:
for i in non_zero_df.columns:
    sns.histplot(data=non_zero_df, x=i, kde=True)
    plt.show()

### we find the outlier to remove the extreme or irrevalent values. however the outlier analysis and chart shows that some of the values are highly right skewed across almost all the features. so i will keep them untouched rn. if needed will change them later.

### Skewness check

In [ ]:
skew_df = non_zero_df.skew().sort_values(ascending=False)
skew_col = skew_df[skew_df>2].index
print(skew_col)
for i in skew_col:
    sns.histplot(data=non_zero_df,x=i)
    plt.show()



### lotarea and miscval has + 4 skew. this will be considered during transformation and using model. Tree based model is not affected by the skewness. Also it is worth notable lotarea most of the value are at the central point but a small amount of house extreme right sided values.

### let's dive into count values

In [ ]:
X_train_df[count_features].describe()

In [ ]:
X_train_df[count_features].nunique()

In [ ]:
X_train_df['TotRmsAbvGrd'].value_counts()

In [ ]:
X_train_df[count_features].isnull().sum()

In [ ]:
##nun values
X_train_df[count_features].eq(0).sum()

### end of count features. the workflow Screen → identify potential issue → investigate meaning → determine it's legitimate → move on.
### i tried to find anomoly then checked for null values. then understood zero represents absence of the features rather than missing info

### Ordinal features

In [ ]:
ordinal_features

In [ ]:
X_train_df[ordinal_features].describe()

In [ ]:
X_train_df[ordinal_features].isnull().sum()

In [ ]:
X_train_df[ordinal_features].eq(0).sum()

### Ordinal feature has no zero and no null

In [ ]:
for i in ordinal_features:
    sns.countplot(data=X_train_df, x=i)
    plt.show()

### Categorical features

In [ ]:
X_train_df[categorical_features].describe()

In [ ]:
cat_null = X_train_df[categorical_features].isnull().sum().sort_values(ascending=False)
cat_null[cat_null > 0]

In [ ]:
##ploting them
fig, ax = plt.subplots(
    ncols=4,
    nrows=3,
    figsize=(20,16)
    )

for ax,feature in zip(ax.flatten(), categorical_features):
    sns.countplot(data=X_train_df, x=feature, ax=ax)
    ax.set_title(feature)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.tight_layout()
plt.show()

### As it appears frequency of the feature is extremely uneven. 

### let's move to temporal

In [ ]:
X_train_df[temporal].describe()

In [ ]:
X_train_df[temporal].isnull().sum()

### From the other feature i can see garage related feature has 64 null values. which tells me that 64 houses doesn't have garage. 

In [ ]:
for i in temporal:
    sns.histplot(data=X_train_df, x=i, )
    plt.xticks(rotation = 90)
    plt.tight_layout()
    plt.show()


### try to find inconsistent data

In [ ]:
invalid_remodel = X_train_df[
    X_train_df['YearRemodAdd'] < X_train_df['YearBuilt']
]
invalid_remodel

In [ ]:
invalid_garage = X_train_df[
    X_train_df['GarageYrBlt'].notnull() &
    (X_train_df['GarageYrBlt'] < X_train_df['YearBuilt']) 
]
invalid_garage[['GarageYrBlt', 'YearBuilt', 'YearRemodAdd', 'GarageFinish', 'GarageType', 'GarageQual', 'GarageCond']]

In [ ]:
invalid_sale = X_train_df[
    X_train_df['YrSold'] < X_train_df['YearBuilt']
]
invalid_sale

### Five observations have GarageYrBlt values earlier than YearBuilt. The differences range from 1 to 10 years. These cases may represent pre-existing garages or property reconstruction history rather than data errors. Since there is no clear evidence that the values are invalid, they will be retained.

In [ ]:
X_train_df['MSSubClass'].value_counts()

In [ ]:
X_train_df['MSSubClass'].isnull().sum()

In [ ]:
X_train_df.loc[
    X_train_df["MSSubClass"] == 40,
    ["MSSubClass", "MSZoning", "Neighborhood", "YearBuilt", "GrLivArea"]
]

### MSSubClass = 40 is a rare category with three observations. The observations occur across different neighborhoods and have relatively large above-ground living areas, but no obvious data inconsistency or implausible values was identified. The category will be retained.

### Time to dive into feature to feature correlation

In [ ]:
corr_matrix = X_train_df[continuous_features].corr()
plt.figure(figsize=(20, 15))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.tight_layout()
plt.show()

In [ ]:
high_corr = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
).stack().sort_values(ascending=False)

high_corr[high_corr > 0.6]

### TotalBsmtSF and 1stFlrSF have strong positive correlation (r ≈ 0.82), which is intuitively expected because larger houses often have both larger basements and larger first floors. However, the features represent different physical aspects of the property, so there is no immediate reason to remove either feature.

### Correlation with respect to target

In [ ]:
X_data = X_train_df[continuous_features].copy()
target = y_train_df.copy()
X_full_data = pd.concat([X_data, target], axis=1)
X_full_data.head()

In [ ]:
X_full_data_corr = X_full_data.corr()
target_corr = X_full_data_corr['SalePrice'].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(20, 15))
sns.heatmap(target_corr.to_frame(), annot=True, cmap='coolwarm')
plt.tight_layout()
plt.show()

In [ ]:
target_corr[target_corr > 0.6]

### let's look into temporal features

In [ ]:
temporal_corr = X_train_df[temporal].corr()
temporal_corr

### yearblt and garageyrblt are correlated but not that much

In [ ]:
fig,axes = plt.subplots(nrows=3, ncols=2, figsize=(15, 15))

for i in temporal:
    sns.scatterplot(data=X_train_df[temporal], x=i, y=y_train_df, ax=axes.flatten()[temporal.index(i)])
    axes.flatten()[temporal.index(i)].set_title(f'SalePrice vs {i}')
    axes.flatten()[temporal.index(i)].set_xlabel(i)
    axes.flatten()[temporal.index(i)].set_ylabel('SalePrice')
    axes.flatten()[temporal.index(i)].tick_params(axis='x', rotation=90)
    plt.tight_layout()
plt.show()

### corr with the target

In [ ]:
X_train_df[temporal].corrwith(y_train_df).sort_values(ascending=False)

### mosold and yrsold are not much correlated with the target

In [ ]:
features = ['MoSold', 'YrSold']
for i in features:
    sns.boxplot(data=X_train_df[temporal], x=i, y=y_train_df)
    plt.show()

### From Temporal YearBuilt       0.516501
YearRemodAdd    0.508593
GarageYrBlt     0.480351 those feature are little bit correlated with the targer

### Ordinal features

In [ ]:
X_train_df[ordinal_features].corrwith(y_train_df).sort_values(ascending=False)

#### Ffs i have found another discovery, there are other feature which are ordinal but i kept them in categorical

In [ ]:
ordinal_features, categorical_features,

In [ ]:
new_ordinal = [ 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','GarageFinish', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
total_ordinal = ordinal_features + new_ordinal
total_ordinal

In [ ]:
modified_categorical_features = [feature for feature in categorical_features if feature not in new_ordinal]
modified_categorical_features, len(categorical_features) , len(modified_categorical_features), len(total_ordinal)

In [ ]:
fig, ax = plt.subplots(
    ncols=3,
    nrows=3,
    figsize=(20,16)
    )
for ax,feature in zip(ax.flatten(), total_ordinal):
    sns.boxplot(data=X_train_df, x=feature, y=y_train_df, ax=ax)
    ax.set_title(feature)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.tight_layout()
plt.show()

### As it apparent that excellent condition houses are tend to be sold at better price. where it is expected most of the houses are scattered around average price

### Count feature

In [ ]:
count_features, len(count_features)

In [ ]:
fig, ax = plt.subplots(
    ncols=3,
    nrows=3,
    figsize=(20,16)
    )

for ax,feature in zip(ax.flatten(), count_features):
    sns.countplot(data=X_train_df[count_features], x=feature, ax=ax)
    ax.set_title(feature)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(
    ncols=3,
    nrows=3,
    figsize=(20,16)
    )
for ax,feature in zip(ax.flatten(), count_features):
    sns.boxplot(data=X_train_df[count_features], x=feature, y=y_train_df, ax=ax)
    ax.set_title(feature)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.tight_layout()
plt.show()

### totalromms above ground - the higher the romm number the more the saleprice, same with fullbath, garagecars more means more money.
### Count features generally show sensible relationships with SalePrice. Some features, such as TotRmsAbvGrd, FullBath, and GarageCars, show a clear positive association, while KitchenAbvGr and BedroomAbvGr exhibit more irregular or non-linear relationships. No count features showed obvious data-quality issues requiring correction or removal.

### Categorical 

In [ ]:
modified_categorical_features, len(modified_categorical_features)  

In [ ]:
fig, ax = plt.subplots(
    ncols=4,
    nrows=4,
    figsize=(20,16)
    )
for ax,feature in zip(ax.flatten(), modified_categorical_features):
    sns.countplot(data=X_train_df[modified_categorical_features], x=feature, ax=ax)
    ax.set_title(feature)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
for start in range(0, len(modified_categorical_features), 6):

    features = modified_categorical_features[start:start+6]

    fig, axes = plt.subplots(
        nrows=3,
        ncols=2,
        figsize=(18, 20)
    )

    for ax, feature in zip(axes.flatten(), features):

        sns.boxplot(
            data=X_train_df,
            x=feature,
            y=y_train_df,
            ax=ax
        )

        ax.set_title(feature)
        ax.tick_params(axis='x', rotation=90)

    plt.tight_layout()
    plt.show()

### Street and LotConfig show relatively similar SalePrice distributions across categories, suggesting a weaker marginal association with the target compared with features such as Neighborhood, LandSlope, and LotShape.alley.
### Utilities is highly imbalanced, with most observations belonging to a single category. The minority categories are very rare but contain valid observations. Therefore, the feature will be retained rather than removed solely due to category imbalance.

### The naked eye fromm the graph doesn't help much. Let's do some statistics. let's find the anova of the categorical features. It's important bc it finds the variance within group is significantly different than the variance of the between group. Why it matters. bc if anova is less than 0.05. 
### ANOVA asks:

"Are the group means statistically significantly different?" (f = between vari/ within var)

Eta-squared asks:

"How much of the total variation is associated with the grouping?"(eta_squared = between var/ total var)

In [ ]:
from scipy.stats import f_oneway
result = []

for feature in modified_categorical_features:
    temporary_df = pd.DataFrame({
        'feature': X_train_df[feature],
        'SalesPrice': y_train_df
    }).dropna()

    groups = [
        group['SalesPrice'].values
        for _, group in temporary_df.groupby('feature')
    ]

    f_test, p_test = f_oneway(*groups)
    result.append({
        'Feature': feature,
        'F-stat' : f_test,
        'P-stat': p_test
    })

result = pd.DataFrame(result)
pd.set_option("display.float_format", "{:.15f}".format)
result.sort_values('P-stat')







Boxplot → understand the actual distributions
↓
F-value / p-value (ANOVA) → statistical evidence of group differences
↓
η² → effect size, i.e. how much variation in SalePrice is associated with the categorical feature

### Anova means that group are different. but eta squared will tell how much varication with respect to the categorical feature

In [ ]:
eta_squared = []

for feature in modified_categorical_features:
    temp = pd.DataFrame({
        'feature': X_train_df[feature],
        'SalesPrice': y_train_df
    }).dropna()

    overall_mean = temp['SalesPrice'].mean()

    group_stat = (
        temp.groupby('feature')['SalesPrice']
        .agg(['mean','count'])
        )

    between_var =( group_stat['count'] *
    (group_stat['mean']- overall_mean) **2 ).sum()

    total_var = ((temp['SalesPrice'] - overall_mean)**2).sum()

    eta = between_var / total_var
    eta_squared.append({
        'Feature': feature,
        'Eta_squared': eta,
        'n_cat': temp['feature'].nunique()
    })

eta_squared = pd.DataFrame(eta_squared)
eta_squared.sort_values('Eta_squared', ascending=False)

### Higher η² means a stronger association between that categorical feature and SalePrice.

### MiscFeature shows differences in the central tendency of SalePrice across categories, supporting an association with the target. However, the feature is highly imbalanced, with some categories having very few observations, so the strength of this association should be interpreted cautiously.

In [ ]:
continuous_features

In [ ]:
X_train_df[continuous_features].isnull().sum()

In [ ]:
X_train_df[temporal].isnull().sum()

In [ ]:
f = X_train_df[modified_categorical_features].isnull().sum()
f[f>0], f

In [ ]:
X_train_df[new_ordinal].isnull().sum()

In [ ]:
X_train_df[count_features].isnull().sum()

In [ ]:
X_train_df[X_train_df['Fireplaces'] == 0].shape

In [ ]:
full_df = X_train_df.copy()
full_df['SalesPrice'] = y_train_df
full_df.head()

In [ ]:
p = full_df.groupby('Neighborhood')['LotFrontage'].mean().sort_values(ascending=False)
print(p)

In [ ]:
X_train_df['MasVnrType'].value_counts()

In [ ]:
X_train_df['MasVnrType'].isnull().sum(), X_train_df['MasVnrArea'].isnull().sum()

In [ ]:
X_train_df.groupby('Neighborhood')['LotFrontage'].median()

In [ ]:
cols = [
    "Id",
    "BsmtCond", "BsmtQual", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath"
]

print(
    train_df.loc[[660, 728], cols].to_string()
)